In [ ]:
import pandas as pd
import numpy as np
import random
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from sklearn.decomposition import PCA
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.optimizers import Adam
import tensorflow as tf
import seaborn as sns
from sklearn.ensemble import IsolationForest
os.chdir('Resources/')

In [ ]:
df = pd.read_csv('5_Preprocessed_Data.csv')

X = df.drop(['HeartDisease'], axis='columns')
Y = df[['HeartDisease']]

In [18]:
seed = 1737

mx=0.9886524643254732
ind=0

for i in range(2, 5000):
    #seed = i
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.9, random_state=491)

    clf = IsolationForest(n_estimators=26, contamination=0.08, random_state=1737, max_samples=256)
    outliers = clf.fit_predict(X_train)
    X_train = X_train[outliers == 1]
    Y_train = Y_train[outliers == 1]

    corr_matrix = X_train.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
    X_train_selected = X_train.drop(to_drop, axis=1)
    X_test_selected = X_test.drop(to_drop, axis=1)

    rf_classifier = RandomForestClassifier(n_estimators=62, random_state=1737, criterion='gini')
    rf_classifier.fit(X_train_selected, Y_train.values.ravel())

    Y_pred = rf_classifier.predict(X_test_selected)

    accuracy_isolation = roc_auc_score(Y_test, Y_pred)

    if(accuracy_isolation>mx):
        mx=accuracy_isolation
        ind=i
    print(i, accuracy_isolation, mx, ind)

194 0.9710808622609048 0.9886524643254732 0
195 0.9710808622609048 0.9886524643254732 0
196 0.9710808622609048 0.9886524643254732 0
197 0.9710808622609048 0.9886524643254732 0
198 0.9710808622609048 0.9886524643254732 0
199 0.9710808622609048 0.9886524643254732 0
200 0.9710808622609048 0.9886524643254732 0
201 0.9710808622609048 0.9886524643254732 0
202 0.9710808622609048 0.9886524643254732 0
203 0.9710808622609048 0.9886524643254732 0
204 0.9710808622609048 0.9886524643254732 0


KeyboardInterrupt: 

In [ ]:
print(mx, ind)

In [ ]:
train_sizes = np.linspace(0.1, 0.9, 9)
AUCs_isolation = []

for i, train_size in enumerate(train_sizes):
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=train_size, random_state=491)

    clf = IsolationForest(n_estimators=26, contamination=0.08, random_state=1737, max_samples=256)
    outliers = clf.fit_predict(X_train)
    X_train = X_train[outliers == 1]
    Y_train = Y_train[outliers == 1]

    corr_matrix = X_train.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
    X_train_selected = X_train.drop(to_drop, axis=1)
    X_test_selected = X_test.drop(to_drop, axis=1)

    rf_isolation = RandomForestClassifier(n_estimators=62, random_state=1737)
    rf_isolation.fit(X_train_selected, Y_train.values.ravel())

    Y_pred = rf_isolation.predict(X_test_selected)
    auc_isolation = roc_auc_score(Y_test, Y_pred)

    AUCs_isolation.append(auc_isolation)

mean_auc_isolation = np.mean(AUCs_isolation)
if mean_auc_isolation>=0.8904262:
    print(f"Mean AUC: {mean_auc_isolation}", "Yes")
else:
    print(f"Mean AUC: {mean_auc_isolation}", "No")